In [1]:
import os

from sklearn.cluster import HDBSCAN
from umap import UMAP

from entity import DatasetManager, TextDatasets, DatasetFactory
from utils import evaluate, random_subset
import optuna

os.chdir("..")

/Users/florian/dev/uni/python/vMF-HAC/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
umap = UMAP(n_components=15, metric="cosine", random_state=42)
dataset_manager = DatasetManager("intfloat/multilingual-e5-large")
dataset = dataset_manager.get(TextDatasets.CLUSTREC_COVID)
x, y = random_subset(dataset.embeddings, dataset.labels, 1000, seed=2)
x_red = umap.fit_transform(x)

/Users/florian/dev/uni/python/vMF-HAC/.venv/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [3]:
hdbscan = HDBSCAN(min_cluster_size=5)
res = evaluate(hdbscan, "clustrec_covid", x_red, y)
res


/Users/florian/dev/uni/python/vMF-HAC/.venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(


ExperimentResult(clusterer_name='HDBSCAN', dataset_name='clustrec_covid', dim=15, nmi=0.7121871695573813, ari=0.4413305162438672, ami=0.5710724831358038, v_measure=0.7121871695573813, noise_fraction=0.323, homogeneity=0.6985997388432744, completeness=0.7263136220415197, fowlkes_mallows=0.4572782588721993, silhouette=0.5213078260421753, calinski_harabasz=478.34186967778106, davies_bouldin=0.6602718807551959, n_clusters=45, n_true_clusters=50, running_time_s=0.009072125001694076, params={'algorithm': 'auto', 'allow_single_cluster': False, 'alpha': 1.0, 'cluster_selection_epsilon': 0.0, 'cluster_selection_method': 'eom', 'copy': 'warn', 'leaf_size': 40, 'max_cluster_size': None, 'metric': 'euclidean', 'metric_params': None, 'min_cluster_size': 5, 'min_samples': None, 'n_jobs': None, 'store_centers': None})

In [5]:
def find_hdbscan_params(dataset_id):
    dataset = dataset_manager.get(dataset_id)
    x, y = random_subset(dataset.embeddings, dataset.labels, min(1000, dataset.labels.shape[0]), seed=2)
    x_red = umap.fit_transform(x)


    def search(trial):
        min_cluster_size = trial.suggest_int("min_cluster_size", 2, 100)
        hdbscan = HDBSCAN(min_cluster_size=min_cluster_size)
        res = evaluate(hdbscan, "clustrec_covid", x_red, y)
        cluster_diff = res.n_clusters - res.n_true_clusters
        if cluster_diff < 0:
            cluster_diff = cluster_diff * -1
        cluster_diff = cluster_diff / res.n_true_clusters
        return res.v_measure * (1 - res.noise_fraction) * (1 - cluster_diff)


    study = optuna.create_study(direction="maximize")
    study.optimize(search, n_trials=100, timeout=60)
    return study.best_params

import json

config = json.load(open("config.json"))
results = []

for dataset_name in config["datasets"]:
    dataset_id = DatasetFactory.from_string(dataset_name)
    best_params = find_hdbscan_params(dataset_id)
    print(f"Best params for {dataset_name}: {best_params}")

    hdbscan = HDBSCAN(min_cluster_size=best_params["min_cluster_size"])
    res = evaluate(hdbscan, "clustrec_covid", x_red, y)
    results.append({"dataset": dataset_name, "best_params": best_params, "v_measure": res.v_measure, "noise_fraction": res.noise_fraction, "n_clusters": res.n_clusters, "n_true_clusters": res.n_true_clusters, "scaled_v_measure": res.v_measure * (1 - res.noise_fraction)})

[I 2026-09-14 11:49:18,409] A new study created in memory with name: no-name-5e3ebcdc-1508-4606-b9f2-d2c903361ca2
/Users/florian/dev/uni/python/vMF-HAC/.venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
[I 2026-09-14 11:49:18,430] Trial 0 finished with value: 0.004990791765892331 and parameters: {'min_cluster_size': 34}. Best is trial 0 with value: 0.004990791765892331.
/Users/florian/dev/uni/python/vMF-HAC/.venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
[I 2026-09-14 11:49:18,453] Trial 1 finished with value: 0.004136853257668176 and parameters: {'min_cluster_size': 97}. Best is trial 0 with value: 0.004990791765892331.
/Users/florian/dev

Best params for Uri-ka/ClusTREC-Covid: {'min_cluster_size': 4}


[I 2026-09-14 11:49:21,961] A new study created in memory with name: no-name-f732f33c-d8a0-4994-9d99-56004f85da20
/Users/florian/dev/uni/python/vMF-HAC/.venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
[I 2026-09-14 11:49:21,976] Trial 0 finished with value: 0.033267187836404685 and parameters: {'min_cluster_size': 18}. Best is trial 0 with value: 0.033267187836404685.
/Users/florian/dev/uni/python/vMF-HAC/.venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
[I 2026-09-14 11:49:21,991] Trial 1 finished with value: 0.0552885127153527 and parameters: {'min_cluster_size': 12}. Best is trial 1 with value: 0.0552885127153527.
/Users/florian/dev/uni

Best params for mteb/biorxiv-clustering-p2p: {'min_cluster_size': 5}


[I 2026-09-14 11:49:25,354] A new study created in memory with name: no-name-ad320f0e-2ac6-46f4-a5c0-c4e01e22b4be
/Users/florian/dev/uni/python/vMF-HAC/.venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
[I 2026-09-14 11:49:25,374] Trial 0 finished with value: 0.0028628954413907354 and parameters: {'min_cluster_size': 55}. Best is trial 0 with value: 0.0028628954413907354.
/Users/florian/dev/uni/python/vMF-HAC/.venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
[I 2026-09-14 11:49:25,395] Trial 1 finished with value: 0.002843069792719529 and parameters: {'min_cluster_size': 76}. Best is trial 0 with value: 0.0028628954413907354.
/Users/florian/

...

Best params for mteb/reddit-clustering: {'min_cluster_size': 4}


[I 2026-09-14 11:50:37,810] A new study created in memory with name: no-name-29978a5e-3a81-4edd-95f7-1cfb6d0bfa22
/Users/florian/dev/uni/python/vMF-HAC/.venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
[I 2026-09-14 11:50:37,832] Trial 0 finished with value: 0.001550461761693415 and parameters: {'min_cluster_size': 66}. Best is trial 0 with value: 0.001550461761693415.
/Users/florian/dev/uni/python/vMF-HAC/.venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
[I 2026-09-14 11:50:37,854] Trial 1 finished with value: 0.0015276008333022428 and parameters: {'min_cluster_size': 90}. Best is trial 0 with value: 0.001550461761693415.
/Users/florian/de

Best params for mteb/reddit-clustering-p2p: {'min_cluster_size': 2}


[I 2026-09-14 11:50:41,684] A new study created in memory with name: no-name-cbf2b416-d056-4aa9-8bd9-44de79f98aa9
/Users/florian/dev/uni/python/vMF-HAC/.venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
[I 2026-09-14 11:50:41,703] Trial 0 finished with value: 0.14840790701416573 and parameters: {'min_cluster_size': 39}. Best is trial 0 with value: 0.14840790701416573.
/Users/florian/dev/uni/python/vMF-HAC/.venv/lib/python3.14/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
[I 2026-09-14 11:50:41,719] Trial 1 finished with value: 0.479625168962313 and parameters: {'min_cluster_size': 26}. Best is trial 1 with value: 0.479625168962313.
/Users/florian/dev/uni/pyt

Best params for mteb/DigikalamagClustering: {'min_cluster_size': 23}


In [7]:
import pandas as pd
pd.DataFrame(results).to_csv("hdbscan_parameters.csv", index=False)

In [ ]:
hdbscan = HDBSCAN(min_cluster_size=study.best_params["min_cluster_size"])
res = evaluate(hdbscan, "clustrec_covid", x_red, y)
res

In [ ]:
(1-res.noise_fraction) * res.v_measure